<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/aiagent.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Day 1 Lab4</h2>
            <span style="color:#ff7800;">Agents Orchestration:<br/>
            This is the fourth lab.<br/>
            We are going to write an agent that sends emails.<br/>
            But, we want a pro style email which sometimes requires reading and re-reading it to make sure if it's okay; It's quite a hassle!<br/>
            Note: this lab only drafts emails.
            </span>
        </td>
    </tr>
</table>

## Orchestration

Our first Agentic Framework project...!!!


### Orchestrating by Code

### Orchestrating by LLMs (skipped in this lab)

-  via Tools
-  via Handoffs

## Agent Orchestration

By and large, there are 2 approacehs for Agent Orchestration; by code and by LLMs.

By code: more predictable and deterministic.

By LLMs: more powerful.

An excellent write-up is here:

https://openai.github.io/openai-agents-python/multi_agent/

We will start with orchestration by code.


In [1]:
# importing the necessary libraries
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent
import asyncio

In [2]:
# loading the environment variables

load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

## Orchestrating by Code

In [3]:
intro = """
You are a sales agent working for a wireless vendor company called WirelessTeleComm, 
a company that provides wireless infrastructures for point-to-point and point-to-multipoint links.
You write emails.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [4]:
# creating three agents with different instructions

sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)


In [ ]:
# To test the agents' performance, one will run in streaming mode.
# To validate further, you can replace sales_agent1 with sales_agent2 or sales_agent3 and see the difference.

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

In [ ]:
message = "Write a cold sales email"  # One prompt for all three: write a cold sales email

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


In [7]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales_picker", instructions=decision, model=MODEL_NAME)


In [ ]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Now go and check out the trace:

https://platform.openai.com/traces